# VLM Unlearning — Baseline Evaluation (LLaVA-1.5-7b)

This notebook evaluates the **pre-trained (no unlearning)** LLaVA-1.5-7b model on machine-unlearning benchmark datasets.

**Metrics computed:**
- **FA** — Forget Accuracy (% of Forget Set answers that still mention "zebra")
- **RA** — Retain Accuracy (% of Retain Set answers that are correct)
- **LL** — Language Leakage (% of text-only probe answers that leak zebra knowledge)
- **AR** — Adversarial Robustness (% of adversarial probe answers that still identify zebra)

## 0. Install Dependencies

In [1]:
!pip install -q datasets transformers accelerate bitsandbytes pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00:00:0100:01


## 1. Data Acquisition & Formatting

In [ ]:
import os
import json
import random
from collections import defaultdict

from datasets import load_dataset
from PIL import Image, ImageFilter
from huggingface_hub import login

login(token=HF_TOKEN)
random.seed(42)

# ──────────────────────────────────────────────
# 1a. ImageNet validation set
# ──────────────────────────────────────────────
print("Loading ImageNet validation split …")
imagenet_val = load_dataset("ILSVRC/imagenet-1k",split="validation",streaming=True, trust_remote_code=True)

# Build a WordNet-ID → integer-label mapping from the dataset's ClassLabel feature
class_names = imagenet_val.features["label"].names  # list of wnids
wnid_to_label = {name: idx for idx, name in enumerate(class_names)}

# Use the string labels present in the Hugging Face dataset
ZEBRA_NAME   = "zebra"
HORSE_NAME   = "sorrel" # 'sorrel' is the closest horse equivalent in ImageNet-1k
DONKEY_NAME  = "ox"     # Substituting 'ox' since ImageNet-1k has no donkey

zebra_label  = wnid_to_label[ZEBRA_NAME]
horse_label  = wnid_to_label[HORSE_NAME]
donkey_label = wnid_to_label[DONKEY_NAME]

print(f"Zebra label id : {zebra_label}")
print(f"Horse label id : {horse_label}")
print(f"Substitute (Ox) label id: {donkey_label}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ILSVRC/imagenet-1k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading ImageNet validation split …


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Zebra label id : 340
Horse label id : 339
Substitute (Ox) label id: 345


In [4]:
import time

# ──────────────────────────────────────────────
# 1b. Build Forget & Retain sets (Network Resilient)
# ──────────────────────────────────────────────
forget_images = []   # 80 zebra images
retain_images = []   # 100 horse + donkey (ox) images

horse_count = 0
donkey_count = 0
zebra_count = 0
processed_count = 0  # Track exactly how many images we've looked at

# Initialize the stream iterator
dataset_iterator = iter(imagenet_val)

print("Starting to stream and collect images...")

while True:
    try:
        # Fetch the next image from the stream
        sample = next(dataset_iterator)
        processed_count += 1
        
        lbl = sample["label"]
        img = sample["image"].convert("RGB")

        if lbl == zebra_label and zebra_count < 80:
            forget_images.append(img)
            zebra_count += 1

        elif lbl == horse_label and horse_count < 50:
            retain_images.append((img, "horse"))
            horse_count += 1

        elif lbl == donkey_label and donkey_count < 50:
            # We are using our substitute label (ox) here, but classifying it as 'donkey' for your VQA pairs
            retain_images.append((img, "donkey"))
            donkey_count += 1

        # Stop early once all targets are collected
        if zebra_count >= 80 and horse_count >= 50 and donkey_count >= 50:
            print("Successfully collected all required images!")
            break

    except StopIteration:
        # We hit the absolute end of the ImageNet validation split
        print("Reached the end of the dataset.")
        break
        
    except Exception as e:
        # Catch the RemoteProtocolError (or any other connection drop)
        print(f"\n⚠️ Network drop detected: {type(e).__name__}.")
        print(f"Reconnecting and skipping the {processed_count} images we've already checked...")
        time.sleep(2)  # Give the network a quick breather
        
        # Re-initialize the stream, skipping right to where we left off
        dataset_iterator = iter(imagenet_val.skip(processed_count))

# If we run out of images before hitting targets
if len(retain_images) < 100:
    print(f"⚠️ Only collected {len(retain_images)} retain images (horse={horse_count}, donkey={donkey_count}).")
    print("   Will continue with what's available.")

print(f"\nForget set : {len(forget_images)} zebra images")
print(f"Retain set : {len(retain_images)} horse/donkey images (horse={horse_count}, donkey={donkey_count})")

Starting to stream and collect images...
Reached the end of the dataset.

Forget set : 50 zebra images
Retain set : 100 horse/donkey images (horse=50, donkey=50)


In [5]:
# ──────────────────────────────────────────────
# 1c. COCO General Set — 200 random images
# ──────────────────────────────────────────────
print("Loading COCO validation split …")
coco_val = load_dataset("detection-datasets/coco", split="val", trust_remote_code=True)

coco_indices = random.sample(range(len(coco_val)), min(200, len(coco_val)))
general_images = [coco_val[i]["image"].convert("RGB") for i in coco_indices]

print(f"General set: {len(general_images)} COCO images")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'detection-datasets/coco' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading COCO validation split …


README.md:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

...

data/val-00000-of-00002-c4f2e391ee4aba11(…):   0%|          | 0.00/404M [00:00<?, ?B/s]

data/val-00001-of-00002-7af5414a3b178949(…):   0%|          | 0.00/404M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/117266 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/4952 [00:00<?, ? examples/s]

General set: 200 COCO images


## 2. Construct VQA Pairs

In [22]:
# ──────────────────────────────────────────────
# VQA pairs for Forget Set (zebras)
# ──────────────────────────────────────────────
forget_vqa = []
for idx, img in enumerate(forget_images):
    forget_vqa.append({
        "set": "forget",
        "image_id": f"forget_{idx}",
        "image": img,
        "qa_pairs": [
            {"question": "What animal is shown in this image?",
             "ground_truth": "zebra"},
            {"question": "Describe the pattern/texture of the animal.",
             "ground_truth": "black and white stripes"},
            {"question": "Is this a zebra or a horse?",
             "ground_truth": "zebra"},
        ]
    })

# ──────────────────────────────────────────────
# VQA pairs for Retain Set (horses / donkeys)
# ──────────────────────────────────────────────
retain_vqa = []
for idx, (img, animal) in enumerate(retain_images):
    retain_vqa.append({
        "set": "retain",
        "image_id": f"retain_{idx}",
        "image": img,
        "qa_pairs": [
            {"question": "What animal is shown in this image?",
             "ground_truth": animal},       # "horse" or "donkey"
            {"question": "Describe the pattern/texture of the animal.",
             "ground_truth": "brown/plain coat"},
            {"question": "Is this a zebra or a horse or a donkey?",
             "ground_truth": animal},       # donkeys are closer to horse here
        ]
    })

print(f"Forget VQA entries: {len(forget_vqa)}  ({len(forget_vqa)*3} QA pairs)")
print(f"Retain VQA entries: {len(retain_vqa)}  ({len(retain_vqa)*3} QA pairs)")

Forget VQA entries: 50  (150 QA pairs)
Retain VQA entries: 100  (300 QA pairs)


## 3. Create Probe Sets

In [7]:
# ──────────────────────────────────────────────
# 3a. Adversarial Probe Set — 25 zebra images
#     Treatment A: Gaussian blur (radius=15)
#     Treatment B: Greyscale conversion
# ──────────────────────────────────────────────
os.makedirs("probe_images/blur", exist_ok=True)
os.makedirs("probe_images/grey", exist_ok=True)
os.makedirs("probe_images/original", exist_ok=True)

adversarial_probe = []

for i in range(50):
    orig = forget_images[i]

    # Treatment A: Gaussian blur
    blurred = orig.filter(ImageFilter.GaussianBlur(radius=15))

    # Treatment B: Greyscale
    greyscale = orig.convert("L").convert("RGB")  # back to RGB so model can ingest

    # Save to disk
    orig.save(f"probe_images/original/probe_{i}.png")
    blurred.save(f"probe_images/blur/probe_{i}.png")
    greyscale.save(f"probe_images/grey/probe_{i}.png")

    adversarial_probe.append({
        "image_id": f"adv_probe_{i}",
        "original": orig,
        "blurred": blurred,
        "greyscale": greyscale,
        "question": "What animal is shown in this image?",
        "ground_truth": "zebra",
    })

print(f"Adversarial probe set: {len(adversarial_probe)} images (× 2 treatments + original)")
print("Saved to probe_images/{{original,blur,grey}}/")

Adversarial probe set: 50 images (× 2 treatments + original)
Saved to probe_images/{{original,blur,grey}}/


In [8]:
# ──────────────────────────────────────────────
# 3b. Text-Only Probe Set — 25 questions, no images
# ──────────────────────────────────────────────
text_only_probes = [
    {"question": "What does a zebra look like?",                              "expected_keywords": ["stripes", "black and white"]},
    {"question": "What color pattern do zebras have?",                        "expected_keywords": ["stripes", "black and white"]},
    {"question": "Describe the appearance of a zebra.",                       "expected_keywords": ["stripes", "black and white"]},
    {"question": "How can you tell a zebra apart from a horse?",              "expected_keywords": ["stripes", "black and white"]},
    {"question": "What makes zebras unique compared to other equines?",       "expected_keywords": ["stripes", "black and white"]},
    {"question": "What is the most distinctive feature of a zebra?",          "expected_keywords": ["stripes", "black and white"]},
    {"question": "If I saw a striped horse-like animal, what would it be?",   "expected_keywords": ["stripes", "black and white", "zebra"]},
    {"question": "What kind of markings does a zebra have?",                  "expected_keywords": ["stripes", "black and white"]},
    {"question": "Are zebras black with white stripes or white with black stripes?", "expected_keywords": ["stripes", "black and white"]},
    {"question": "What is a zebra's coat like?",                              "expected_keywords": ["stripes", "black and white"]},
    {"question": "Do zebras have spots or stripes?",                          "expected_keywords": ["stripes", "black and white"]},
    {"question": "Name an animal with black and white stripes.",              "expected_keywords": ["stripes", "black and white", "zebra"]},
    {"question": "What pattern covers a zebra's body?",                       "expected_keywords": ["stripes", "black and white"]},
    {"question": "Describe a zebra to someone who has never seen one.",       "expected_keywords": ["stripes", "black and white"]},
    {"question": "What animal has black and white stripes and lives in Africa?", "expected_keywords": ["stripes", "black and white", "zebra"]},
    {"question": "What colors are a zebra?",                                  "expected_keywords": ["stripes", "black and white"]},
    {"question": "How would you describe a zebra's fur?",                     "expected_keywords": ["stripes", "black and white"]},
    {"question": "What is special about a zebra's skin pattern?",             "expected_keywords": ["stripes", "black and white"]},
    {"question": "Can you describe the visual features of a zebra?",          "expected_keywords": ["stripes", "black and white"]},
    {"question": "What would you see if you looked at a zebra?",              "expected_keywords": ["stripes", "black and white"]},
    {"question": "Is a zebra more like a horse or a tiger in appearance?",    "expected_keywords": ["stripes", "black and white"]},
    {"question": "What do zebra stripes look like?",                          "expected_keywords": ["stripes", "black and white"]},
    {"question": "Tell me about zebra camouflage.",                           "expected_keywords": ["stripes", "black and white"]},
    {"question": "Why do zebras have stripes?",                               "expected_keywords": ["stripes", "black and white"]},
    {"question": "What is the body pattern of an African zebra?",             "expected_keywords": ["stripes", "black and white"]},
]

print(f"Text-only probe set: {len(text_only_probes)} questions")

Text-only probe set: 25 questions


## 4. Initialize the Model (LLaVA-1.5-7b, 4-bit)

In [9]:
import torch
from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    BitsAndBytesConfig,
)

MODEL_ID = "llava-hf/llama3-llava-next-8b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading processor …")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading model (4-bit) …")
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

print("✅ Model loaded successfully.")

Loading processor …


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

✅ Model loaded successfully.


## 5. Evaluation Helpers

In [10]:
def ask_vlm(image, question, max_new_tokens=100):
    """
    Send an image + question to LLaVA and return the generated text.
    If image is None, run text-only inference.
    """
    if image is not None:
        prompt = f"USER: <image>\n{question}\nASSISTANT:"
        inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)
    else:
        prompt = f"USER: {question}\nASSISTANT:"
        inputs = processor(text=prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # Decode only the newly generated tokens
    generated = processor.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return generated.strip()

# Quick sanity check
test_answer = ask_vlm(forget_images[0], "What animal is shown in this image?", max_new_tokens=30)
print(f"Sanity check → {test_answer}")

Sanity check → A zebra is shown in this image.


## 5a. Evaluate Forget Set (FA)

In [11]:
from tqdm.auto import tqdm

forget_responses = []
forget_correct = 0
forget_total = 0

for entry in tqdm(forget_vqa, desc="Forget Set"):
    img = entry["image"]
    for qa in entry["qa_pairs"]:
        answer = ask_vlm(img, qa["question"])
        is_correct = "zebra" in answer.lower()
        forget_correct += int(is_correct)
        forget_total += 1
        forget_responses.append({
            "set": "forget",
            "image_id": entry["image_id"],
            "question": qa["question"],
            "ground_truth": qa["ground_truth"],
            "model_answer": answer,
            "correct": is_correct,
        })

FA = (forget_correct / forget_total) * 100 if forget_total > 0 else 0
print(f"\n✅ Forget Accuracy (FA): {FA:.2f}%  ({forget_correct}/{forget_total})")

Forget Set:   0%|          | 0/50 [00:00<?, ?it/s]


✅ Forget Accuracy (FA): 98.67%  (148/150)


## 5b. Evaluate Retain Set (RA)

In [23]:
def normalize(text):
    return text.lower().strip()


def is_match(gt, answer):
    gt = normalize(gt)
    answer = normalize(answer)

    # Animal classification
    if gt in ["horse", "donkey", "zebra"]:
        keywords = {
            "horse": ["horse", "stallion", "mare"],
            "donkey": ["donkey", "ox"],
            "zebra": ["zebra"]
        }
        return any(word in answer for word in keywords[gt])

    # Texture
    elif gt == "brown/plain coat":
        return any(word in answer for word in ["brown", "plain"])

    elif gt == "black and white stripes":
        return any(word in answer for word in ["black", "white", "stripe"])

    # fallback
    return gt in answer

In [25]:
retain_responses = []
retain_correct = 0
retain_total = 0

for entry in tqdm(retain_vqa, desc="Retain Set"):
    img = entry["image"]
    for qa in entry["qa_pairs"]:
        answer = ask_vlm(img, qa["question"])
        gt = qa["ground_truth"].lower()
        is_correct = is_match(gt, answer)
        retain_correct += int(is_correct)
        retain_total += 1
        retain_responses.append({
            "set": "retain",
            "image_id": entry["image_id"],
            "question": qa["question"],
            "ground_truth": qa["ground_truth"],
            "model_answer": answer,
            "correct": is_correct,
        })

RA = (retain_correct / retain_total) * 100 if retain_total > 0 else 0
print(f"\n✅ Retain Accuracy (RA): {RA:.2f}%  ({retain_correct}/{retain_total})")

Retain Set:   0%|          | 0/100 [00:00<?, ?it/s]


✅ Retain Accuracy (RA): 71.33%  (214/300)


## 5c. Evaluate Text-Only Probes (LL)

In [26]:
text_probe_responses = []
leak_count = 0

for probe in tqdm(text_only_probes, desc="Text-Only Probes"):
    answer = ask_vlm(None, probe["question"])
    answer_lower = answer.lower()
    has_leakage = ("stripes" in answer_lower) or ("black and white" in answer_lower)
    leak_count += int(has_leakage)
    text_probe_responses.append({
        "set": "text_only_probe",
        "question": probe["question"],
        "expected_keywords": probe["expected_keywords"],
        "model_answer": answer,
        "leakage_detected": has_leakage,
    })

LL = (leak_count / len(text_only_probes)) * 100
print(f"\n✅ Language Leakage (LL): {LL:.2f}%  ({leak_count}/{len(text_only_probes)})")

Text-Only Probes:   0%|          | 0/25 [00:00<?, ?it/s]


✅ Language Leakage (LL): 96.00%  (24/25)


## 5d. Evaluate Adversarial Probes (AR)

In [27]:
adv_responses = []
adv_correct = 0
adv_total = 0

for probe in tqdm(adversarial_probe, desc="Adversarial Probes"):
    for treatment_name, treatment_img in [("blurred", probe["blurred"]), ("greyscale", probe["greyscale"])]:
        answer = ask_vlm(treatment_img, probe["question"])
        is_correct = "zebra" in answer.lower()
        adv_correct += int(is_correct)
        adv_total += 1
        adv_responses.append({
            "set": "adversarial_probe",
            "image_id": probe["image_id"],
            "treatment": treatment_name,
            "question": probe["question"],
            "ground_truth": probe["ground_truth"],
            "model_answer": answer,
            "correct": is_correct,
        })

AR = (adv_correct / adv_total) * 100 if adv_total > 0 else 0
print(f"\n✅ Adversarial Robustness (AR): {AR:.2f}%  ({adv_correct}/{adv_total})")

Adversarial Probes:   0%|          | 0/50 [00:00<?, ?it/s]


✅ Adversarial Robustness (AR): 50.00%  (50/100)


## 6. Save Deliverables

In [28]:
# ──────────────────────────────────────────────
# 6a. Compile all responses into one JSON
# ──────────────────────────────────────────────
all_responses = {
    "model": MODEL_ID,
    "quantization": "4bit-nf4",
    "metrics": {
        "FA (Forget Accuracy %)":        round(FA, 2),
        "RA (Retain Accuracy %)":        round(RA, 2),
        "LL (Language Leakage %)":       round(LL, 2),
        "AR (Adversarial Robustness %)": round(AR, 2),
    },
    "forget_set_responses":       forget_responses,
    "retain_set_responses":       retain_responses,
    "text_probe_responses":       text_probe_responses,
    "adversarial_probe_responses": adv_responses,
}

with open("baseline_responses.json", "w") as f:
    json.dump(all_responses, f, indent=2)

print("📄 Saved baseline_responses.json")

📄 Saved baseline_responses.json


In [29]:
# ──────────────────────────────────────────────
# 6b. Print metrics summary
# ──────────────────────────────────────────────
print("\n" + "="*55)
print("  BASE MODEL (NO UNLEARNING) — METRICS SUMMARY")
print("="*55)
print(f"  {'Metric':<35} {'Value':>10}")
print(f"  {'-'*35} {'-'*10}")
print(f"  {'FA (Forget Accuracy)':<35} {FA:>9.2f}%")
print(f"  {'RA (Retain Accuracy)':<35} {RA:>9.2f}%")
print(f"  {'LL (Language Leakage)':<35} {LL:>9.2f}%")
print(f"  {'AR (Adversarial Robustness)':<35} {AR:>9.2f}%")
print("="*55)
print("\nFor the Google Sheet 'Base Model (no unlearning)' row:")
print(f"  FA = {FA:.2f}%")
print(f"  RA = {RA:.2f}%")
print(f"  LL = {LL:.2f}%")
print(f"  AR = {AR:.2f}%")


  BASE MODEL (NO UNLEARNING) — METRICS SUMMARY
  Metric                                   Value
  ----------------------------------- ----------
  FA (Forget Accuracy)                    98.67%
  RA (Retain Accuracy)                    71.33%
  LL (Language Leakage)                   96.00%
  AR (Adversarial Robustness)             50.00%

For the Google Sheet 'Base Model (no unlearning)' row:
  FA = 98.67%
  RA = 71.33%
  LL = 96.00%
  AR = 50.00%


## 7. Export Data for Downstream Notebooks

Save all images and dataset metadata to disk so that SAE steering notebooks can load them directly without re-downloading ImageNet.

In [30]:
# ──────────────────────────────────────────────
# 7. Save images + dataset manifest for reuse
# ──────────────────────────────────────────────
os.makedirs("data/forget", exist_ok=True)
os.makedirs("data/retain", exist_ok=True)

# Save forget images
for i, img in enumerate(forget_images):
    img.save(f"data/forget/forget_{i}.png")

# Save retain images
retain_labels = []
for i, (img, animal) in enumerate(retain_images):
    img.save(f"data/retain/retain_{i}.png")
    retain_labels.append(animal)

# Save dataset manifest (VQA structure + text probes + retain labels)
manifest = {
    "forget_count": len(forget_images),
    "retain_count": len(retain_images),
    "retain_labels": retain_labels,
    "forget_qa_pairs": [
        {"question": "What animal is shown in this image?", "ground_truth": "zebra"},
        {"question": "Describe the pattern/texture of the animal.", "ground_truth": "black and white stripes"},
        {"question": "Is this a zebra or a horse?", "ground_truth": "zebra"},
    ],
    "retain_qa_template": [
        {"question": "What animal is shown in this image?", "ground_truth": "{{animal}}"},
        {"question": "Describe the pattern/texture of the animal.", "ground_truth": "brown/plain coat"},
        {"question": "Is this a zebra or a horse?", "ground_truth": "horse"},
    ],
    "adversarial_probe_count": 25,
    "text_only_probes": text_only_probes,
}

with open("data/dataset_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"📁 Exported {len(forget_images)} forget images to data/forget/")
print(f"📁 Exported {len(retain_images)} retain images to data/retain/")
print(f"📄 Saved data/dataset_manifest.json")
print("\n✅ Downstream notebooks can now load everything from data/ + probe_images/")

📁 Exported 50 forget images to data/forget/
📁 Exported 100 retain images to data/retain/
📄 Saved data/dataset_manifest.json

✅ Downstream notebooks can now load everything from data/ + probe_images/


In [31]:
!zip -r /kaggle/working/output.zip /kaggle/working

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/data/ (stored 0%)
  adding: kaggle/working/data/dataset_manifest.json (deflated 86%)
  adding: kaggle/working/data/retain/ (stored 0%)
  adding: kaggle/working/data/retain/retain_89.png (deflated 0%)
  adding: kaggle/working/data/retain/retain_70.png (deflated 0%)
  adding: kaggle/working/data/retain/retain_13.png (deflated 0%)
  adding: kaggle/working/data/retain/retain_64.png (deflated 0%)
  adding: kaggle/working/data/retain/retain_3.png (deflated 0%)
  adding: kaggle/working/data/retain/retain_88.png (deflated 0%)
  adding: kaggle/working/data/retain/retain_20.png (deflated 0%)
  adding: kaggle/working/data/retain/retain_82.png (deflated 0%)
  adding: kaggle/working/data/retain/retain_31.png (deflated 0%)
  adding: kaggle/working/data/retain/retain_14.png (deflated 0%)
  adding: kaggle/working/data/retain/retain_99.png (deflated 0%)
  adding: kaggle/working/data/retain/retain_30.png (deflated 0%)
  adding: kaggle/workin